# 01 · Profiling the raw Favorita files

**Project:** Shelfcast, a grocery demand planning system   
**Goal:** look at every raw file before designing any tables or models. For each file we check: shape, dtypes, nulls, the unique key, the date range, and anything weird.    
**Data:** [Corporación Favorita Grocery Sales Forecasting (Kaggle)](https://www.kaggle.com/competitions/favorita-grocery-sales-forecasting/data). The CSVs live in `data/raw/` and are not committed to git.     
**Column meanings and sources:** see `docs/data_dictionary.md`.     

## Setup

In [ ]:
import pandas as pd
pd.set_option('display.expand_frame_repr', False)

`profile(df, name)` runs the same checks on any table: shape, dtypes, null counts, unique values per column, and the first rows.

Tip: if a column's unique count equals the number of rows, that column alone identifies each row, so it is a candidate key.

In [ ]:
def profile(df, name):
    print('name: ', name)
    print()
    print('shape: ', df.shape)
    print()
    print('dtypes: ')
    print(df.dtypes)
    print()
    print('null values: ')
    print(df.isnull().sum())
    print()
    print('unique values: ')
    print(df.nunique())
    print()
    print('first few rows: ')
    print(df.head())

## 1. stores.csv: store metadata

One row per store: where it is (city, state) and how Favorita groups it (type, cluster).

In [ ]:
stores = pd.read_csv('../data/raw/stores.csv')
profile(stores, 'stores')

name:  stores

shape:  (54, 5)

dtypes: 
store_nbr    int64
city           str
state          str
type           str
cluster      int64
dtype: object

null values: 
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

unique values: 
store_nbr    54
city         22
state        16
type          5
cluster      17
dtype: int64

first few rows: 
   store_nbr           city                           state type  cluster
0          1          Quito                       Pichincha    D       13
1          2          Quito                       Pichincha    D       13
2          3          Quito                       Pichincha    D        8
3          4          Quito                       Pichincha    D        9
4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4


How are the 54 stores spread across cities, states, types and clusters?

In [ ]:
print(stores['city'].value_counts())
print(stores['state'].value_counts())
print(stores['type'].value_counts())
print(stores['cluster'].value_counts())

city
Quito            18
Guayaquil         8
Santo Domingo     3
Cuenca            3
Latacunga         2
Ambato            2
Machala           2
Manta             2
Cayambe           1
Riobamba          1
Ibarra            1
Guaranda          1
Puyo              1
Salinas           1
Daule             1
Babahoyo          1
Quevedo           1
Playas            1
Libertad          1
Loja              1
Esmeraldas        1
El Carmen         1
Name: count, dtype: int64
state
Pichincha                         19
Guayas                            11
Santo Domingo de los Tsachilas     3
Azuay                              3
Manabi                             3
Cotopaxi                           2
Tungurahua                         2
Los Rios                           2
El Oro                             2
Chimborazo                         1
Imbabura                           1
Bolivar                            1
Pastaza                            1
Santa Elena                        1
Loja 

### Findings: stores.csv

- **Shape:** 54 rows × 5 columns, no nulls.
- **Key:** `store_nbr`. 54 unique values in 54 rows, so one row per store.
- **Cities:** 22. Quito has 18 stores (a third of the chain), Guayaquil 8, and 14 cities have only 1 store.
- **States:** 16 (Ecuador's provinces). Pichincha has 19 stores, Guayas 11.
- **type:** 5 values (D 18, C 15, A 9, B 8, E 4). Favorita never documented what the letters mean, so we treat it as a plain label.
- **cluster:** 17 values with 1 to 7 stores each. Officially "a grouping of similar stores", but how it was built is not documented.


## 2. items.csv: product metadata

One row per item: its product family, its class, and whether it is perishable.

In [ ]:
items = pd.read_csv('../data/raw/items.csv')
profile(items, 'items')

name:  items

shape:  (4100, 4)

dtypes: 
item_nbr      int64
family          str
class         int64
perishable    int64
dtype: object

null values: 
item_nbr      0
family        0
class         0
perishable    0
dtype: int64

unique values: 
item_nbr      4100
family          33
class          337
perishable       2
dtype: int64

first few rows: 
   item_nbr        family  class  perishable
0     96995     GROCERY I   1093           0
1     99197     GROCERY I   1067           0
2    103501      CLEANING   3008           0
3    103520     GROCERY I   1028           0
4    103665  BREAD/BAKERY   2712           1


How are the 4,100 items spread across families, and how many are perishable?

In [ ]:
print(items['family'].value_counts())
print(items['perishable'].value_counts())

family
GROCERY I                     1334
BEVERAGES                      613
CLEANING                       446
PRODUCE                        306
DAIRY                          242
PERSONAL CARE                  153
BREAD/BAKERY                   134
HOME CARE                      108
DELI                            91
MEATS                           84
HOME AND KITCHEN I              77
LIQUOR,WINE,BEER                73
FROZEN FOODS                    55
POULTRY                         54
HOME AND KITCHEN II             45
EGGS                            41
CELEBRATION                     31
PREPARED FOODS                  26
LAWN AND GARDEN                 26
LADIESWEAR                      21
LINGERIE                        20
AUTOMOTIVE                      20
BEAUTY                          19
PLAYERS AND ELECTRONICS         17
SCHOOL AND OFFICE SUPPLIES      15
GROCERY II                      14
PET SUPPLIES                    14
SEAFOOD                          8
MAGAZINES    

Is there a hierarchy? Does each class belong to only one family?

In [ ]:
class_families = items.groupby('class')['family'].nunique()
class_families.max()

np.int64(1)

family -> class -> item

Do all class codes have the same number of digits?

In [ ]:
print(items['class'].min())
print(items['class'].max())

1002
7780


Do the first two digits of `class` match the family? Checked in both directions.

In [ ]:
items['class_prefix'] = items['class'] // 100

prefix_families = items.groupby('class_prefix')['family'].nunique()
print(prefix_families.max())

family_prefixes = items.groupby('family')['class_prefix'].nunique()
print(family_prefixes.max())

1
1


### Findings: items.csv

- **Shape:** 4,100 rows × 4 columns, no nulls.
- **Key:** `item_nbr`. 4,100 unique values in 4,100 rows, so one row per item. IDs are large codes (e.g. 96995), not 1 to 4100.
- **family:** 33 values. GROCERY I is the largest (1,334 items, about a third), then BEVERAGES (613) and CLEANING (446). HOME APPLIANCES, BABY CARE and BOOKS have 1 item each. Includes non-food families (e.g. LADIESWEAR, AUTOMOTIVE).
- **class:** 337 values. Codes only (1002 to 7780, all 4 digits), no class names given.
- **perishable:** 0/1 flag. 986 items (about 24%) are perishable.
- **Hierarchy:** family → class → item (checked). Every class belongs to exactly one family, and each item has one row, so exactly one class.
- **Class codes:** the first two digits (`class // 100`) map one-to-one to family (checked). Example: 1093 → 10 = GROCERY I.

## 3. oil.csv: daily oil price

One row per trading day: the WTI crude oil price in US dollars per barrel.

In [ ]:
oil = pd.read_csv('../data/raw/oil.csv')
profile(oil, 'oil')

name:  oil

shape:  (1218, 2)

dtypes: 
date              str
dcoilwtico    float64
dtype: object

null values: 
date           0
dcoilwtico    43
dtype: int64

unique values: 
date          1218
dcoilwtico     998
dtype: int64

first few rows: 
         date  dcoilwtico
0  2013-01-01         NaN
1  2013-01-02       93.14
2  2013-01-03       92.97
3  2013-01-04       93.12
4  2013-01-07       93.20


What date range does oil cover?

In [ ]:
print(oil['date'].min())
print(oil['date'].max())
oil['date'] = pd.to_datetime(oil['date'])
print(oil['date'].max()-oil['date'].min()   )

2013-01-01
2017-08-31
1703 days 00:00:00


Are weekends recorded? Which days of the week appear?

In [ ]:
print(oil['date'].dt.day_name().value_counts(dropna=False))

date
Tuesday      244
Wednesday    244
Thursday     244
Friday       243
Monday       243
Name: count, dtype: int64


Which dates have no price?

In [ ]:
null_oil = oil[oil['dcoilwtico'].isnull()]
print(null_oil)

           date  dcoilwtico
0    2013-01-01         NaN
14   2013-01-21         NaN
34   2013-02-18         NaN
63   2013-03-29         NaN
104  2013-05-27         NaN
132  2013-07-04         NaN
174  2013-09-02         NaN
237  2013-11-28         NaN
256  2013-12-25         NaN
261  2014-01-01         NaN
274  2014-01-20         NaN
294  2014-02-17         NaN
338  2014-04-18         NaN
364  2014-05-26         NaN
393  2014-07-04         NaN
434  2014-09-01         NaN
497  2014-11-27         NaN
517  2014-12-25         NaN
522  2015-01-01         NaN
534  2015-01-19         NaN
554  2015-02-16         NaN
588  2015-04-03         NaN
624  2015-05-25         NaN
653  2015-07-03         NaN
699  2015-09-07         NaN
757  2015-11-26         NaN
778  2015-12-25         NaN
783  2016-01-01         NaN
794  2016-01-18         NaN
814  2016-02-15         NaN
843  2016-03-25         NaN
889  2016-05-30         NaN
914  2016-07-04         NaN
959  2016-09-05         NaN
1017 2016-11-24     

### Findings: oil.csv

- **Shape:** 1,218 rows × 2 columns.
- **Key:** `date`. 1,218 unique values in 1,218 rows, so one row per date.
- **Range:** 2013-01-01 to 2017-08-31.
- **Days:** weekdays only, Monday to Friday with 243–244 rows each. Weekends have no row.
- **dcoilwtico:** WTI crude oil spot price in US dollars per barrel (the name matches FRED series DCOILWTICO). 43 nulls.
- **Null prices:** 42 fall on US market holidays (NYSE calendar), including Good Friday and weekend holidays observed on the nearest weekday (e.g. 2016-12-26, 2017-01-02). 1 is unexplained: 2017-07-03, an early-close day, not a full holiday.
- **Two kinds of gaps:** weekends have no row at all, while market holidays have a row with a null price.

## 4. holidays_events.csv: holidays and events

One row per holiday or event (not per day): when it happens, what kind of day it is, and where it applies (country, province or city).

In [ ]:
holidays_events = pd.read_csv('../data/raw/holidays_events.csv')
profile(holidays_events, 'holidays_events.csv')

name:  holidays_events.csv

shape:  (350, 6)

dtypes: 
date            str
type            str
locale          str
locale_name     str
description     str
transferred    bool
dtype: object

null values: 
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

unique values: 
date           312
type             6
locale           3
locale_name     24
description    103
transferred      2
dtype: int64

first few rows: 
         date     type    locale locale_name                    description  \
0  2012-03-02  Holiday     Local       Manta             Fundacion de Manta   
1  2012-04-01  Holiday  Regional    Cotopaxi  Provincializacion de Cotopaxi   
2  2012-04-12  Holiday     Local      Cuenca            Fundacion de Cuenca   
3  2012-04-14  Holiday     Local    Libertad      Cantonizacion de Libertad   
4  2012-04-21  Holiday     Local    Riobamba      Cantonizacion de Riobamba   

   transferred  
0        False  
1        F

What date range does it cover?

In [ ]:
holidays_events['date'] = pd.to_datetime(holidays_events['date'])

In [ ]:
print(holidays_events['date'].max())
print(holidays_events['date'].min())

2017-12-26 00:00:00
2012-03-02 00:00:00


How are the rows spread across types, levels and places?

In [ ]:
for col in ['type', 'locale', 'locale_name', 'transferred']:
    print(f" \n Value counts for '{col}':")
    print(holidays_events[col].value_counts(dropna=False))

 
 Value counts for 'type':
type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64
 
 Value counts for 'locale':
locale
National    174
Local       152
Regional     24
Name: count, dtype: int64
 
 Value counts for 'locale_name':
locale_name
Ecuador                           174
Quito                              13
Riobamba                           12
Guaranda                           12
Latacunga                          12
Ambato                             12
Guayaquil                          11
Cuenca                              7
Ibarra                              7
Manta                               6
Cotopaxi                            6
Libertad                            6
Puyo                                6
Imbabura                            6
Machala                             6
Santo Domingo                       6
El Carmen                           6
Cayambe                      

Do the 12 `transferred = True` rows pair with the 12 `type = Transfer` rows?

In [ ]:
holidays_events['day'] = holidays_events['date'].dt.day_name()
print(holidays_events[holidays_events['transferred'] == True])


          date     type    locale locale_name                    description  transferred        day
19  2012-10-09  Holiday  National     Ecuador     Independencia de Guayaquil         True    Tuesday
72  2013-10-09  Holiday  National     Ecuador     Independencia de Guayaquil         True  Wednesday
135 2014-10-09  Holiday  National     Ecuador     Independencia de Guayaquil         True   Thursday
255 2016-05-24  Holiday  National     Ecuador           Batalla de Pichincha         True    Tuesday
266 2016-07-25  Holiday     Local   Guayaquil         Fundacion de Guayaquil         True     Monday
268 2016-08-10  Holiday  National     Ecuador  Primer Grito de Independencia         True  Wednesday
297 2017-01-01  Holiday  National     Ecuador             Primer dia del ano         True     Sunday
303 2017-04-12  Holiday     Local      Cuenca            Fundacion de Cuenca         True  Wednesday
312 2017-05-24  Holiday  National     Ecuador           Batalla de Pichincha         True  

In [ ]:
print(holidays_events[holidays_events['type'] == 'Transfer'])

          date      type    locale locale_name                             description  transferred       day
20  2012-10-12  Transfer  National     Ecuador     Traslado Independencia de Guayaquil        False    Friday
73  2013-10-11  Transfer  National     Ecuador     Traslado Independencia de Guayaquil        False    Friday
136 2014-10-10  Transfer  National     Ecuador     Traslado Independencia de Guayaquil        False    Friday
256 2016-05-27  Transfer  National     Ecuador           Traslado Batalla de Pichincha        False    Friday
265 2016-07-24  Transfer     Local   Guayaquil         Traslado Fundacion de Guayaquil        False    Sunday
269 2016-08-12  Transfer  National     Ecuador  Traslado Primer Grito de Independencia        False    Friday
298 2017-01-02  Transfer  National     Ecuador             Traslado Primer dia del ano        False    Monday
304 2017-04-13  Transfer     Local      Cuenca                     Fundacion de Cuenca        False  Thursday
313 2017-0

Do the 5 Bridge rows pair with the 5 Work Day rows?

In [ ]:
holidays_events[holidays_events['type'] == 'Bridge']

,date,type,locale,locale_name,description,transferred,day
35,2012-12-24,Bridge,National,Ecuador,Puente Navidad,False,Monday
39,2012-12-31,Bridge,National,Ecuador,Puente Primer dia del ano,False,Monday
156,2014-12-26,Bridge,National,Ecuador,Puente Navidad,False,Friday
160,2015-01-02,Bridge,National,Ecuador,Puente Primer dia del ano,False,Friday
277,2016-11-04,Bridge,National,Ecuador,Puente Dia de Difuntos,False,Friday


In [ ]:
holidays_events[holidays_events['type'] == 'Work Day']

,date,type,locale,locale_name,description,transferred,day
42,2013-01-05,Work Day,National,Ecuador,Recupero puente Navidad,False,Saturday
43,2013-01-12,Work Day,National,Ecuador,Recupero puente primer dia del ano,False,Saturday
149,2014-12-20,Work Day,National,Ecuador,Recupero Puente Navidad,False,Saturday
161,2015-01-10,Work Day,National,Ecuador,Recupero Puente Primer dia del ano,False,Saturday
283,2016-11-12,Work Day,National,Ecuador,Recupero Puente Dia de Difuntos,False,Saturday


Is `date` + `locale_name` unique?

In [ ]:
holidays_events.duplicated(subset=['date', 'locale_name']).sum()

np.int64(7)

Which rows share the same date and place?

In [ ]:
print(holidays_events[holidays_events.duplicated(subset=['date', 'locale_name'], keep=False)])

          date        type    locale locale_name                      description  transferred       day
35  2012-12-24      Bridge  National     Ecuador                   Puente Navidad        False    Monday
36  2012-12-24  Additional  National     Ecuador                        Navidad-1        False    Monday
39  2012-12-31      Bridge  National     Ecuador        Puente Primer dia del ano        False    Monday
40  2012-12-31  Additional  National     Ecuador             Primer dia del ano-1        False    Monday
156 2014-12-26      Bridge  National     Ecuador                   Puente Navidad        False    Friday
157 2014-12-26  Additional  National     Ecuador                        Navidad+1        False    Friday
235 2016-05-01     Holiday  National     Ecuador                  Dia del Trabajo        False    Sunday
236 2016-05-01       Event  National     Ecuador              Terremoto Manabi+15        False    Sunday
242 2016-05-07  Additional  National     Ecuador       

Is `date` + `locale_name` + `description` unique?

In [ ]:
holidays_events.duplicated(subset=['date', 'locale_name', 'description']).sum()

np.int64(0)

In [ ]:
holidays_events.head()

,date,type,locale,locale_name,description,transferred,day
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False,Friday
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False,Sunday
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False,Thursday
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False,Saturday
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False,Saturday


Does every `locale_name` match a city or state in stores (or Ecuador)?

In [ ]:
valid_places = list(stores['city']) + list(stores['state']) + ['Ecuador']
matched = holidays_events['locale_name'].isin(valid_places)
print((~matched).sum())

0


### Findings: holidays_events.csv

- **Shape:** 350 rows × 6 columns, no nulls.
- **Key:** `date` + `locale_name` + `description` (0 duplicates). One row per holiday or event, not per day: only 312 unique dates.
- **Range:** 2012-03-02 to 2017-12-26.
- **type:** Holiday 221, Event 56, Additional 51, Transfer 12, Bridge 5, Work Day 5.
- **locale / locale_name:** the level and the place. National 174 (all `Ecuador`), Regional 24 (provinces), Local 152 (cities). Every `locale_name` matches a stores city, a stores state, or Ecuador (checked).
- **Transfers:** the 12 `transferred = True` rows (official date, worked as a normal day) pair with the 12 `type = Transfer` rows (the day actually taken off), same place, 1 to 3 days apart. 9 of the 12 were moved to a Friday.
- **Bridges:** the 5 Bridge days (Monday or Friday, "Puente") pair with 5 Work Days (all Saturdays, "Recupero"), worked before or after the bridge to pay it back.
- **Same date and place:** 7 pairs. Bridge + Additional (3), earthquake event "Terremoto Manabi+N" + another holiday or event (3), Guayaquil 2016-07-24 Additional + Transfer (1).
- **Oddities:** 2 Transfer rows (Cuenca 2017-04-13, Ibarra 2017-09-29) lack the "Traslado" prefix. Fundación de Guayaquil was moved from Monday 2016-07-25 to Sunday 2016-07-24, unexplained.

## 5. transactions.csv: daily transactions per store

One row per store per day: how many sales transactions that store had.

In [ ]:
transactions = pd.read_csv('../data/raw/transactions.csv')
profile(transactions, 'transactions')

name:  transactions

shape:  (83488, 3)

dtypes: 
date              str
store_nbr       int64
transactions    int64
dtype: object

null values: 
date            0
store_nbr       0
transactions    0
dtype: int64

unique values: 
date            1682
store_nbr         54
transactions    4993
dtype: int64

first few rows: 
         date  store_nbr  transactions
0  2013-01-01         25           770
1  2013-01-02          1          2111
2  2013-01-02          2          2358
3  2013-01-02          3          3487
4  2013-01-02          4          1922


What date range does it cover?

In [ ]:
transactions['date'] = pd.to_datetime(transactions['date'])
print(transactions['date'].min())
print(transactions['date'].max())

2013-01-01 00:00:00
2017-08-15 00:00:00


How many days is that?

In [ ]:
transactions['date'].max() - transactions['date'].min()

Timedelta('1687 days 00:00:00')

Are any days missing completely?

In [ ]:
all_days = pd.date_range(transactions['date'].min(), transactions['date'].max())
missing_days = all_days[~all_days.isin(transactions['date'])]
print(missing_days)

DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-01-01',
               '2016-01-03', '2016-12-25'],
              dtype='datetime64[us]', freq=None)


Is 2016-01-03 a holiday?

In [ ]:
print(pd.Timestamp('2016-01-03').day_name())
pd.Timestamp('2016-01-03') in holidays_events['date'].values

Sunday


False

Is `date` + `store_nbr` unique?

In [ ]:
transactions.duplicated(subset=['date', 'store_nbr']).sum()

np.int64(0)

How many stores report on each day?

In [ ]:
stores_per_day = transactions.groupby('date')['store_nbr'].nunique()

summary = stores_per_day.value_counts().reset_index()
summary.columns = ['stores_that_day', 'number_of_days']
summary = summary.sort_values('stores_that_day', ascending=False)
print(summary)

    stores_that_day  number_of_days
4                54             118
0                53             440
7                52              91
6                51              97
8                50              65
5                49              97
3                48             164
1                47             406
2                46             197
10               43               1
12               36               1
13               14               1
11                2               1
9                 1               3


Which days have unusually few stores?

In [ ]:
print(stores_per_day[stores_per_day < 45])

date
2013-01-01     1
2013-06-19    43
2014-01-01     2
2015-01-01     1
2016-01-02    36
2016-01-04    14
2017-01-01     1
Name: store_nbr, dtype: int64


When did each store first and last appear?

In [ ]:
store_dates = transactions.groupby('store_nbr')['date'].agg(['min', 'max', 'count'])
print(store_dates.sort_values('min', ascending=False).head(10))

                 min        max  count
store_nbr                             
52        2017-04-20 2017-08-15    118
22        2015-10-09 2017-08-15    671
42        2015-08-21 2017-08-15    720
21        2015-07-24 2017-08-15    748
29        2015-03-20 2017-08-15    874
20        2015-02-13 2017-08-15    909
53        2014-05-29 2017-08-15   1167
36        2013-05-09 2017-08-15   1551
39        2013-01-02 2017-08-15   1678
31        2013-01-02 2017-08-15   1678


### Findings: transactions.csv

- **Shape:** 83,488 rows × 3 columns, no nulls.
- **Key:** `date` + `store_nbr` (0 duplicates).
- **Range:** 2013-01-01 to 2017-08-15 (1,688 days).
- **Missing days:** 6 days have no rows at all: Christmas 2013–2016, 2016-01-01, and 2016-01-03 (a Sunday, not in holidays_events, unexplained).
- **Completeness:** all stores on all days would be 1,682 × 54 = 90,828 rows. 83,488 exist, so about 7,300 store-days are missing.
- **Stores per day:** all 54 on only 118 days. Most days have 53 (440 days) or 47 (406 days).
- **Low days:** New Year's Days have 1–2 stores. 2016-01-02 (36), 2016-01-04 (14) and 2013-06-19 (43) are unusually low, unexplained.
- **Late openings:** store 52 (2017-04-20, 118 days), stores 20, 21, 22, 29, 42 (2015), 53 (2014) and 36 (2013). Every store's last date is 2017-08-15, so none closed.

## 6. train.csv: daily sales

One row per store × item × day with sales. The file is 4.65 GB, so it is loaded with smaller dtypes to save memory.

In [ ]:
train = pd.read_csv(
    '../data/raw/train.csv',
    dtype={
        'id': 'int32',
        'store_nbr': 'int8',
        'item_nbr': 'int32',
        'unit_sales': 'float32',
        'onpromotion': 'object'
    },
    parse_dates=['date'],
)

print(train.memory_usage().sum() / 1e9)

3.639414292


In [ ]:
train.columns

Index(['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion'],
      dtype='str')

In [ ]:
profile(train, 'train')

name:  train

shape:  (125497040, 6)

dtypes: 
id                      int32
date           datetime64[us]
store_nbr                int8
item_nbr                int32
unit_sales            float32
onpromotion            object
dtype: object

null values: 
id                    0
date                  0
store_nbr             0
item_nbr              0
unit_sales            0
onpromotion    21657651
dtype: int64

unique values: 
id             125497040
date                1684
store_nbr             54
item_nbr            4036
unit_sales        258474
onpromotion            2
dtype: int64

first few rows: 
   id       date  store_nbr  item_nbr  unit_sales onpromotion
0   0 2013-01-01         25    103665         7.0         NaN
1   1 2013-01-01         25    105574         1.0         NaN
2   2 2013-01-01         25    105575         2.0         NaN
3   3 2013-01-01         25    108079         1.0         NaN
4   4 2013-01-01         25    108701         1.0         NaN


What date range does it cover?

In [ ]:
print(train['date'].min())
print(train['date'].max())

2013-01-01 00:00:00
2017-08-15 00:00:00


Is `date` + `store_nbr` + `item_nbr` unique?

In [ ]:
train.duplicated(subset=['date', 'store_nbr', 'item_nbr']).sum()

np.int64(0)

Are any days missing completely?

In [ ]:
all_days = pd.date_range(train['date'].min(), train['date'].max())
missing_days = all_days[~all_days.isin(train['date'])]
print(missing_days)

DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'],
              dtype='datetime64[us]', freq=None)


Do sales also drop in early January 2016, where transactions.csv is missing most stores?

In [ ]:
rows_per_day = train['date'].value_counts().sort_index()
print(rows_per_day['2015-12-30':'2016-01-06'])

date
2015-12-30     98252
2015-12-31     96689
2016-01-01      1718
2016-01-02    100830
2016-01-03    100370
2016-01-04     96557
2016-01-05     94461
2016-01-06     93639
Name: count, dtype: int64


What do sales look like? How many are returns (negative) or fractions?

In [ ]:
print(train['unit_sales'].describe())
print((train['unit_sales'] < 0).sum()) # returns
is_fraction = train['unit_sales'] % 1 != 0
print(is_fraction.sum()) # is not whole number

count    1.254970e+08
mean     8.554866e+00
std      2.360515e+01
min     -1.537200e+04
25%      2.000000e+00
50%      4.000000e+00
75%      9.000000e+00
max      8.944000e+04
Name: unit_sales, dtype: float64
7795
8191095


Which rows are the most extreme?

In [ ]:
# the most extreme rows
print(train.loc[train['unit_sales'].idxmax()])
print(train.loc[train['unit_sales'].idxmin()])


id                        93189596
date           2016-10-07 00:00:00
store_nbr                       39
item_nbr                   1976284
unit_sales                 89440.0
onpromotion                  False
Name: 93189596, dtype: object
id                        49592112
date           2015-06-22 00:00:00
store_nbr                       18
item_nbr                   1166474
unit_sales                -15372.0
onpromotion                  False
Name: 49592112, dtype: object


Which families have fractional sales?

In [ ]:
# which families have fractional sales?
fraction_items = train.loc[is_fraction, 'item_nbr'].unique()
print(items[items['item_nbr'].isin(fraction_items)]['family'].value_counts())

family
PRODUCE           128
MEATS              83
POULTRY            54
FROZEN FOODS       13
DELI                8
SEAFOOD             3
PREPARED FOODS      2
GROCERY I           2
BREAD/BAKERY        1
Name: count, dtype: int64


Is the 89,440 sale normal for that item?

In [ ]:
# the 89,440 sale
big = train[(train['store_nbr'] == 39) & (train['item_nbr'] == 1976284)]
print(items[items['item_nbr'] == 1976284])
print(big['unit_sales'].describe())
print(big[(big['date'] >= '2016-10-01') & (big['date'] <= '2016-10-14')])

      item_nbr family  class  perishable
3434   1976284  MEATS   2302           1
count       25.000000
mean      6743.809570
std      17315.083984
min       1049.000000
25%       1836.000000
50%       3817.000000
75%       5021.200195
max      89440.000000
Name: unit_sales, dtype: float64
                id       date  store_nbr  item_nbr  unit_sales onpromotion
93189596  93189596 2016-10-07         39   1976284     89440.0       False


What happened around the −15,372 return?

In [ ]:
# the -15,372 return
ret = train[(train['store_nbr'] == 18) & (train['item_nbr'] == 1166474)]
print(items[items['item_nbr'] == 1166474])
print(ret['unit_sales'].describe())
print(ret[(ret['date'] >= '2015-06-15') & (ret['date'] <= '2015-06-29')])

      item_nbr     family  class  perishable
1773   1166474  BEVERAGES   1120           0
count     1106.000000
mean        13.366184
std        541.375061
min     -15372.000000
25%          5.000000
50%         10.000000
75%         18.000000
max       9025.000000
Name: unit_sales, dtype: float64
                id       date  store_nbr  item_nbr  unit_sales onpromotion
49037408  49037408 2015-06-15         18   1166474         4.0       False
49115873  49115873 2015-06-16         18   1166474         7.0       False
49194329  49194329 2015-06-17         18   1166474         9.0       False
49271596  49271596 2015-06-18         18   1166474         5.0       False
49347648  49347648 2015-06-19         18   1166474      9025.0       False
49427617  49427617 2015-06-20         18   1166474        18.0       False
49510634  49510634 2015-06-21         18   1166474        22.0       False
49592112  49592112 2015-06-22         18   1166474    -15372.0       False
49671039  49671039 2015-06

When is `onpromotion` empty?

In [ ]:
print(train['onpromotion'].value_counts(dropna=False))
promo_missing_dates = train.loc[train['onpromotion'].isnull(), 'date']
print(promo_missing_dates.min())
print(promo_missing_dates.max())

onpromotion
False    96028767
NaN      21657651
True      7810622
Name: count, dtype: int64
2013-01-01 00:00:00
2014-03-31 00:00:00


Is every row before 2014-04-01 empty?

In [ ]:
print(train.loc[train['date'] < '2014-04-01', 'onpromotion'].isnull().mean())

1.0


Does every sold item exist in items.csv?

In [ ]:
print(train['item_nbr'].nunique())
print((~train['item_nbr'].isin(items['item_nbr'])).sum())

4036
0


How many of all possible store × item × day rows exist?

In [ ]:
expected_rows = train['date'].nunique() * train['store_nbr'].nunique() * train['item_nbr'].nunique()
print(expected_rows)
print(len(train))

367017696
125497040


### Findings: train.csv

- **Shape:** 125,497,040 rows × 6 columns. Loaded with smaller dtypes, it takes 3.6 GB of memory.
- **Key:** `date` + `store_nbr` + `item_nbr` (0 duplicates).
- **Range:** 2013-01-01 to 2017-08-15. Only the 4 Christmases (2013–2016) have no rows.
- **Early January 2016:** 2016-01-01 has 1,718 rows (New Year). 2016-01-02 to 01-04 have normal counts (about 97,000–101,000), although transactions.csv is missing most stores on those days.
- **unit_sales:** median 4, 75% of rows 9 or less, mean 8.55. Range −15,372 to 89,440.
- **Returns:** 7,795 negative rows.
- **Fractions:** 8,191,095 rows (6.5%) are not whole numbers, all in weighed families: PRODUCE, MEATS, POULTRY, FROZEN FOODS, DELI, SEAFOOD, PREPARED FOODS, plus 2 GROCERY I items and 1 BREAD/BAKERY item.
- **Extremes:** the max, 89,440 (store 39, MEATS item 1976284, 2016-10-07), belongs to an item sold on only 25 days, always 1,049 or more. The min, −15,372 (store 18, BEVERAGES item 1166474, 2015-06-22), comes 3 days after a 9,025 spike; the item normally sells about 10 a day. Cause unknown.
- **onpromotion:** False 96,028,767, True 7,810,622, empty 21,657,651 (17.3%). Every row before 2014-04-01 is empty and none after, so promotions were not recorded before then.
- **Items:** 4,036 items sold, all of them in items.csv. 64 catalogue items never sold.
- **Completeness:** 1,684 days × 54 stores × 4,036 items = 367,017,696 possible rows. 125,497,040 exist (34%).